# Multi-Seed RealMLP Ensemble
Runs RealMLP with 3 different seeds and blends predictions for maximum diversity.

In [ ]:
import math, random, warnings, numpy as np, pandas as pd, os
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore')
# Use CPU to avoid CUDA issues on Kaggle (more reliable)
device = torch.device('cpu')
print(f'PyTorch: {torch.__version__} | Device: {device} (CPU for reliability)')

In [ ]:
import os
# Find the correct data path
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'train.csv':
            data_dir = root
            print(f'Found data at: {data_dir}')
            break
train = pd.read_csv(f'{data_dir}/train.csv')
test = pd.read_csv(f'{data_dir}/test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')
ID, TARGET = 'id', 'class'
train[TARGET] = train[TARGET].map({'GALAXY':0,'QSO':1,'STAR':2})
train_id = train[ID]; test_id = test[ID]; y = train[TARGET]; n_classes = 3

In [ ]:
category_map = {}
color_pairs = [('u','g'),('u','r')]
important_combos = sorted([('alpha_cat_','delta_cat_'),('u_cat_','z_cat_')])

def feature_engineering(df, fit=False, cat_cols_ref=None, num_cols_ref=None):
    df = df.copy()
    df['_g_/_redshift'] = (df['g']/(df['redshift']+1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i']/(df['redshift']+1e-6)).astype('float32')
    for a,b in color_pairs: df[f'_{a}-{b}'] = (df[a]-df[b]).astype('float32')
    if fit:
        cat_cols_ref = df.select_dtypes(include=['object']).columns.tolist()
        num_cols_ref = df.select_dtypes(exclude=['object']).columns.tolist()
    for col in cat_cols_ref:
        if fit: codes,uniques=df[col].factorize(); category_map[col]=uniques
        else: uniques=category_map[col]; cm={c:i for i,c in enumerate(uniques)}; codes=df[col].map(cm).fillna(-1).astype('int32')
        df[col]=codes; df[col]=df[col].astype('category')
    for col in num_cols_ref:
        cn=f'{col}_cat_'
        if fit: codes,uniques=np.floor(df[col]).factorize(); category_map[col]=uniques
        else: uniques=category_map[col]; cm={c:i for i,c in enumerate(uniques)}; codes=np.floor(df[col]).map(cm).fillna(-1).astype('int32')
        df[cn]=codes; df[cn]=df[cn].astype('category')
    for col,bins_l in {'delta':[100,500]}.items():
        for nb in bins_l:
            bn=f'{col}_{nb}_quantile_bin_'
            if fit: kb=KBinsDiscretizer(n_bins=nb,encode='ordinal',strategy='quantile',subsample=None); binned=kb.fit_transform(df[[col]]).ravel().astype('int32'); category_map[bn]=kb
            else: kb=category_map[bn]; binned=kb.transform(df[[col]]).ravel().astype('int32')
            df[bn]=binned; df[bn]=df[bn].astype('category')
    combo_names=[]
    for cols in important_combos:
        cn='_'.join(cols)+'_'; combo_names.append(cn); cs=df[cols[0]].astype(str)
        for c in cols[1:]: cs=cs+'_'+df[c].astype(str)
        if fit: codes,uniques=pd.factorize(cs,sort=False); category_map[cn]=uniques
        else: uniques=category_map[cn]; cm={c:i for i,c in enumerate(uniques)}; codes=cs.map(cm).fillna(-1).astype('int32')
        df[cn]=codes; df[cn]=df[cn].astype('category')
    new_cat=[c for c in df.columns if c.endswith('_')]; new_num=[c for c in df.columns if c.startswith('_')]
    return df,cat_cols_ref,num_cols_ref,new_cat,new_num,combo_names

X_raw=train.drop([ID,TARGET],axis=1); X_test_raw=test.drop([ID],axis=1)
X,cat_cols,num_cols,new_cat,new_num,combo_names=feature_engineering(X_raw,fit=True)
X_test,_,_,_,_,_=feature_engineering(X_test_raw,fit=False,cat_cols_ref=cat_cols,num_cols_ref=num_cols)
cat_cols+=new_cat; num_cols+=new_num; cat_cols=sorted(cat_cols)
X=X.reindex(sorted(X.columns),axis=1); X_test=X_test.reindex(sorted(X_test.columns),axis=1)
del train,test,X_raw,X_test_raw
print(f'Features: {X.shape[1]}, Cat: {len(cat_cols)}, Num: {len(num_cols)}')

In [ ]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(s,t): s._tfms=[x for x in t if x in ('median_center','robust_scale','smooth_clip','l2_normalize')]
    def fit(s,X,y=None):
        if 'median_center' in s._tfms or 'robust_scale' in s._tfms:
            s._median=np.median(X,0); q=np.quantile(X,.75,0)-np.quantile(X,.25,0)
            z=q==0.; q[z]=.5*(X.max(0)[z]-X.min(0)[z]); s._iqr=1./(q+1e-30); s._iqr[q==0.]=0.
        return s
    def transform(s,X,y=None):
        X=X.copy().astype('float32')
        for t in s._tfms:
            if t=='median_center': X-=s._median[None,:]
            elif t=='robust_scale': X*=s._iqr[None,:]
            elif t=='smooth_clip': X=X/np.sqrt(1+(X/3)**2)
            elif t=='l2_normalize': n=np.linalg.norm(X,1,keepdims=True); X/=np.where(n==0,1.,n)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(s,n,cd,ed=8,ot=8):
        super().__init__(); s.n=n; s.cd=cd; s.oh=[]; s.el=nn.ModuleList(); s._efi=[]
        for i,d in enumerate(cd):
            if d<=ot: s.oh.append(i)
            else: s.el.append(nn.ModuleList([nn.Embedding(d,ed) for _ in range(n)])); s._efi.append(i)
    def forward(s,x):
        B,N,_=x.shape; fs=[]
        if s.oh:
            ox=x[:,:,s.oh]; od=[s.cd[i] for i in s.oh]; to=sum(od)
            enc=torch.zeros(B,N,to,device=x.device); st=0
            for i,d in enumerate(od): p=ox[:,:,i:i+1].long(); enc.scatter_(2,p+st,1.); st+=d
            fs.append(enc)
        for el,fi in zip(s.el,s._efi):
            fe=[el[j](x[:,j,fi:fi+1].long()) for j in range(s.n)]; fs.append(torch.cat(fe,1))
        return torch.cat(fs,2)

class ScalingLayer(nn.Module):
    def __init__(s,n,f): super().__init__(); s.sc=nn.Parameter(torch.ones(n,f))
    def forward(s,x): return x*s.sc[None,:,:]

class NTPLinear(nn.Module):
    def __init__(s,n,i,o,b=True):
        super().__init__(); s.i=i; s.w=nn.Parameter(torch.randn(n,i,o)); s.b=nn.Parameter(torch.randn(n,o)) if b else None
    def forward(s,x):
        x=torch.einsum('bki,kio->bko',x,s.w)/math.sqrt(s.i)
        if s.b is not None: x=x+s.b
        return x

class PBLDEmbedding(nn.Module):
    def __init__(s,n,f,h=16,o=4,fs=.1,act=nn.GELU):
        super().__init__(); s.n=n; s.f=f; s.o=o
        s.w1=nn.Parameter(torch.randn(n,f,h)*fs); s.b1=nn.Parameter(torch.randn(n,f,h))
        s.w2=nn.Parameter(torch.randn(n,f,h,o-1)/math.sqrt(h)); s.b2=nn.Parameter(torch.zeros(n,f,o-1))
        s.act=act(); nn.init.uniform_(s.b1,-math.pi,math.pi)
    def forward(s,x):
        p=torch.cos(2*math.pi*(x.unsqueeze(-1)*s.w1.unsqueeze(0)+s.b1.unsqueeze(0)))
        t=s.act(torch.einsum('bkfh,kfhd->bkfd',p,s.w2)+s.b2.unsqueeze(0))
        return torch.cat([x.unsqueeze(-1),t],-1).flatten(2)

class RealMLP(nn.Module):
    def __init__(s,od,cd,nn_,cfg):
        super().__init__(); n=cfg['n_ens']; ed=cfg['embed_dim']; s.n=n
        s.cate=CategoricalFeatureLayer(n,cd,ed,cfg['onehot_thresh'])
        s.ne=PBLDEmbedding(n,nn_,cfg['pbld_hidden_dim'],cfg['pbld_out_dim'],cfg['pbld_freq_scale'],cfg['pbld_activation'])
        td=nn_*cfg['pbld_out_dim']+(sum(c if c<=cfg['onehot_thresh'] else ed for c in cd))
        layers=[]; act=cfg['activation']
        if cfg['add_front_scale']: layers.append(ScalingLayer(n,td))
        s._dm=[]; ind=td
        for i,hd in enumerate(cfg['hidden_dims']):
            l=NTPLinear(n,ind,hd)
            if i==0: s.fl=l
            d=nn.Dropout(cfg['dropout']); s._dm.append(d); layers+=[l,act(),d]; ind=hd
        s.h=nn.Sequential(*layers); s.ol=NTPLinear(n,ind,od)
    def forward(s,xn,xc):
        xn=xn.unsqueeze(1).expand(-1,s.n,-1); xc=xc.unsqueeze(1).expand(-1,s.n,-1)
        xn=s.ne(xn); xc=s.cate(xc); x=s.h(torch.cat([xn,xc],2)); x=s.ol(x)
        return F.softmax(x,2)

def apply_sched(iv,p,s,fr=.3):
    if s=='constant': return iv
    elif s=='cos': return iv*(math.cos(math.pi*p)+1)/2
    elif s=='flat_cos':
        if p<fr: return iv
        t=(p-fr)/(1-fr); return iv*(math.cos(math.pi*t)+1)/2
    elif s=='expm4t': return iv*math.exp(-4*p)
    return iv

def get_pg(m,p):
    fid=id(m.fl.w); sp,pp,fw,ow,bp=[],[],[],[],[]
    for n,pm in m.named_parameters():
        if 'ne' in n: pp.append(pm)
        elif 'sc' in n: sp.append(pm)
        elif id(pm)==fid: fw.append(pm)
        elif 'b' in n and 'ne' not in n: bp.append(pm)
        else: ow.append(pm)
    LR=p['lr']; WD=p['weight_decay']
    return [{'params':sp,'lr':LR*p['lr_scale_mult'],'weight_decay':WD*p['wd_scale_mult']},
            {'params':pp,'lr':LR*p['pbld_lr_factor'],'weight_decay':WD},
            {'params':fw,'lr':LR*p['first_layer_lr_factor'],'weight_decay':WD*p['first_layer_wd_factor']},
            {'params':ow,'lr':LR,'weight_decay':WD},
            {'params':bp,'lr':LR*p['lr_bias_mult'],'weight_decay':WD*p['wd_bias_mult']}]

def smooth_ce(yp,yt,ls=0.,cw=None):
    nc=yp.size(1); ys=torch.full_like(yp,ls/nc); ys.scatter_(1,yt.unsqueeze(1),1.-ls+ls/nc)
    pl=-(ys*torch.log(yp.clamp(1e-15,1))).sum(1)
    if cw is not None: sw=cw[yt]; return (pl*sw).sum()/sw.sum()
    return pl.mean()

class RealMLP_TD(BaseEstimator):
    def __init__(s,**kw): s.p={**CONFIG,**kw}
    def fit(s,Xt,yt,Xv,yv,cat_cols=None,ckpt='ck.pth',X_test=None):
        p=s.p; dev=torch.device(p['device'] if torch.cuda.is_available() else 'cpu')
        cc=cat_cols or []; nc=[c for c in Xt.columns if c not in cc]
        Xtn=Xt[nc].values.astype('float32'); Xvn=Xv[nc].values.astype('float32')
        Xtc=Xt[cc].values.astype('int64'); Xvc=Xv[cc].values.astype('int64')
        y_=np.asarray(yt); yv_=np.asarray(yv)
        s.pp=NumericalPreprocessor(p['tfms']).fit(Xtn)
        Xtn=s.pp.transform(Xtn); Xvn=s.pp.transform(Xvn)
        s.cc=cc; s.nc=nc
        if cc:
            ac=[Xtc,Xvc]
            if X_test is not None: ac.append(X_test[cc].values.astype('int64'))
            cd=(np.concatenate(ac,0).max(0)+1).tolist()
        else: cd=[]
        s.cd=cd
        if cd:
            cm=np.array(cd)-1; Xtc=np.clip(Xtc,0,cm); Xvc=np.clip(Xvc,0,cm)
        cls=np.unique(y_); s.cls_=cls
        # FIX: use keyword arguments for newer sklearn
        cw=torch.as_tensor(compute_class_weight(class_weight='balanced',classes=cls,y=y_),dtype=torch.float32,device=dev)
        s.m_=RealMLP(len(cls),cd,Xtn.shape[1],p).to(dev)
        pg=get_pg(s.m_,p)
        for g in pg: g['lr_base']=g['lr']
        opt=torch.optim.AdamW(pg,betas=(p['mom'],p['sq_mom']))
        Xtn_=torch.as_tensor(Xtn,device=dev); Xtc_=torch.as_tensor(Xtc,device=dev)
        ytt=torch.as_tensor(y_,device=dev)
        Xvn_=torch.as_tensor(Xvn,device=dev); Xvc_=torch.as_tensor(Xvc,device=dev)
        ne=p['n_ens']; bs=p['train_bs']; ebs=p['eval_bs']; ep=p['epochs']
        ls=p['lr_sched']; fr=p['flat_ratio']; ts=ep*len(y_)
        to=np.arange(len(y_)); bs_=-np.inf; be=0; bvp=None
        for e in range(ep):
            s.m_.train()
            for i in range(0,len(y_),bs):
                pr=(e*len(y_)+i)/ts; ix=to[i:i+bs]
                for g in opt.param_groups: g['lr']=apply_sched(g['lr_base'],pr,ls,fr)
                opt.zero_grad()
                yp=s.m_(Xtn_[ix],Xtc_[ix])
                lv=apply_sched(p['ls_eps'],pr,p['ls_eps_sched'],fr)
                dv=apply_sched(p['dropout'],pr,p['p_drop_sched'],fr)
                for dm in s.m_._dm: dm.p=dv
                loss=smooth_ce(yp.reshape(-1,len(cls)),ytt[ix].repeat_interleave(ne),lv,cw)
                loss.backward(); torch.nn.utils.clip_grad_norm_(s.m_.parameters(),p['grad_clip']); opt.step()
            np.random.shuffle(to)
            s.m_.eval()
            with torch.no_grad():
                vp=np.concatenate([s.m_(Xvn_[i:i+ebs],Xvc_[i:i+ebs]).mean(1).cpu().numpy() for i in range(0,len(yv_),ebs)])
            sc=balanced_accuracy_score(yv_,np.argmax(vp,1))
            if sc>bs_: bs_=sc; be=e+1; bvp=vp.copy(); torch.save(s.m_.state_dict(),ckpt)
            print(f'  ep {e+1}/{ep} sc={sc:.5f} best={bs_:.5f}'+' *' if sc==bs_ else '')
        s.m_.load_state_dict(torch.load(ckpt,weights_only=True)); s.bs_=bs_; s.bvp=bvp; s._d=dev
        print(f'  -> best: {bs_:.5f} (ep {be})')
        return s
    def predict_proba(s,X):
        ebs=s.p['eval_bs']; Xn=s.pp.transform(X[s.nc].values.astype('float32'))
        Xc=np.clip(X[s.cc].values.astype('int64'),0,np.array(s.cd)-1)
        Xn=torch.as_tensor(Xn,device=s._d); Xc=torch.as_tensor(Xc,device=s._d)
        s.m_.eval()
        with torch.no_grad():
            return np.concatenate([s.m_(Xn[i:i+ebs],Xc[i:i+ebs]).mean(1).cpu().numpy() for i in range(0,len(Xn),ebs)])

In [ ]:
CONFIG = {
    'n_ens':8,'embed_dim':8,'onehot_thresh':8,
    'hidden_dims':[512,512,512],'dropout':0.06,'p_drop_sched':'expm4t',
    'activation':nn.SiLU,'add_front_scale':True,
    'pbld_hidden_dim':20,'pbld_out_dim':5,'pbld_freq_scale':5.0,
    'pbld_activation':nn.PReLU,'pbld_lr_factor':0.093,
    'lr':0.01,'mom':0.9,'sq_mom':0.98,
    'lr_sched':'flat_cos','flat_ratio':0.3,
    'first_layer_lr_factor':1.0,'first_layer_wd_factor':0.1,
    'lr_scale_mult':10.0,'lr_bias_mult':0.1,
    'weight_decay':0.013,'wd_scale_mult':0.1,'wd_bias_mult':0.5,'grad_clip':1.0,
    'ls_eps':0.04,'ls_eps_sched':'cos',
    'tfms':['median_center','robust_scale'],
    'epochs':6,'train_bs':256,'eval_bs':10240,'verbosity':2,
    'use_early_stopping':False,
    'early_stopping_additive_patience':10,'early_stopping_multiplicative_patience':1,
    'device':'cuda','random_state':42,
}
SEEDS = [42, 123, 777]
FOLDS = 5

In [ ]:
all_test = []; all_oof = []
for si,SEED in enumerate(SEEDS):
    print(f'\n{"="*50}\n  SEED={SEED} ({si+1}/{len(SEEDS)})\n{"="*50}')
    np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
    skf=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
    oof=np.zeros((len(X),n_classes)); tst=np.zeros((len(X_test),n_classes))
    for fold,(tr,vl) in enumerate(skf.split(X,y),1):
        Xt=X.iloc[tr].copy(); Xv=X.iloc[vl].copy(); Xs=X_test.copy()
        enc=TargetEncoder(cv=FOLDS,smooth='auto',shuffle=True,random_state=SEED)
        tenc=enc.fit_transform(Xt[combo_names],y.iloc[tr])
        venc=enc.transform(Xv[combo_names]); senc=enc.transform(Xs[combo_names])
        ten=[f'_{c}TE{cls}' for c in combo_names for cls in range(n_classes)]
        Xt[ten]=tenc; Xv[ten]=venc; Xs[ten]=senc
        print(f'### Seed {SEED} | Fold {fold}/{FOLDS}')
        m=RealMLP_TD(**CONFIG)
        m.fit(Xt,y.iloc[tr],Xv,y.iloc[vl],cat_cols=cat_cols,ckpt=f'm_s{SEED}_f{fold}.pth')
        oof[vl]=m.bvp; tst+=m.predict_proba(Xs)/FOLDS
        torch.cuda.empty_cache()
    acc=accuracy_score(y,np.argmax(oof,1))
    print(f'Seed {SEED} OOF Acc: {acc:.5f}')
    all_test.append(tst); all_oof.append(oof)

In [ ]:
# Blend seeds
print('Blending seeds...')
best_acc=0; best_w=None
for w1 in np.arange(0.2,0.6,0.05):
    for w2 in np.arange(0.1,0.5,0.05):
        w3=1-w1-w2
        if w3<0.1: continue
        bl=w1*all_oof[0]+w2*all_oof[1]+w3*all_oof[2]
        a=accuracy_score(y,np.argmax(bl,1))
        if a>best_acc: best_acc=a; best_w=(w1,w2,w3)
print(f'Best weights: {best_w} Acc: {best_acc:.5f}')

# Also try all same weight
avg_acc=accuracy_score(y,np.argmax(sum(all_oof)/3,1))
print(f'Equal weight: {avg_acc:.5f}')

final_tst=best_w[0]*all_test[0]+best_w[1]*all_test[1]+best_w[2]*all_test[2]
sub=pd.DataFrame({ID:test_id,TARGET:np.argmax(final_tst,1)})
sub[TARGET]=sub[TARGET].map({0:'GALAXY',1:'QSO',2:'STAR'})
sub.to_csv('submission.csv',index=False)
print(f'\nSaved submission.csv')
sub.head()